In [21]:
import numpy as np
import pandas as pd

# 為了讓每次亂數結果都一樣
np.random.seed(42)

In [22]:
# -------------------------
# 基本設定
# -------------------------

n_samples = 100 # 100位病人
n_genes = 100 # 100個基因

gene_names = [f"Gene{i}" for i in range(1, n_genes + 1)]

sample_names = [f"P{i}" for i in range(1, n_samples + 1)]

In [23]:
labels = (
    ["Virus"] * 25
    + ["Gram+"] * 25
    + ["Gram-"] * 25
    + ["Non"] * 25
)

In [24]:
expression = np.random.normal( # 正常情況下未感染的基因表現量，服從Normal(5,1)
    loc=5,
    scale=1,
    size=(n_samples, n_genes)
)

In [25]:
print(expression)

[[5.49671415 4.8617357  5.64768854 ... 5.26105527 5.00511346 4.76541287]
 [3.58462926 4.57935468 4.65728548 ... 5.15372511 5.05820872 3.8570297 ]
 [5.35778736 5.56078453 6.08305124 ... 5.30729952 5.81286212 5.62962884]
 ...
 [4.75400466 5.44670361 5.58705753 ... 5.3753597  4.88481167 3.81691448]
 [5.93381043 5.50323238 7.30563684 ... 4.72972963 6.61286837 5.94661831]
 [5.93465471 5.36992561 3.88267299 ... 4.29468328 5.49576557 5.64438845]]


In [26]:
# 調整受感染下，基因表現的情況

virus_idx = np.array(labels) == "Virus" # 感染病毒的25位病人，gene 1 ~ gene 3的表現量會提高三個標準差

expression[virus_idx, 0] += 3
expression[virus_idx, 1] += 3
expression[virus_idx, 2] += 3

gram_pos_idx = np.array(labels) == "Gram+" # 感染革蘭氏陽性菌的25位病人，gene 11 ~ gene 13的表現量會提高三個標準差

expression[gram_pos_idx, 10] += 3
expression[gram_pos_idx, 11] += 3
expression[gram_pos_idx, 12] += 3

gram_neg_idx = np.array(labels) == "Gram-" # 感染革蘭氏陰性菌的25位病人，gene 21 ~ gene 23的表現量會提高三個標準差

expression[gram_neg_idx, 20] += 3
expression[gram_neg_idx, 21] += 3
expression[gram_neg_idx, 22] += 3

In [27]:
print(expression)

[[8.49671415 7.8617357  8.64768854 ... 5.26105527 5.00511346 4.76541287]
 [6.58462926 7.57935468 7.65728548 ... 5.15372511 5.05820872 3.8570297 ]
 [8.35778736 8.56078453 9.08305124 ... 5.30729952 5.81286212 5.62962884]
 ...
 [4.75400466 5.44670361 5.58705753 ... 5.3753597  4.88481167 3.81691448]
 [5.93381043 5.50323238 7.30563684 ... 4.72972963 6.61286837 5.94661831]
 [5.93465471 5.36992561 3.88267299 ... 4.29468328 5.49576557 5.64438845]]


In [28]:
expression_df = pd.DataFrame(
    expression,
    columns=gene_names,
    index=sample_names
)

expression_df["Label"] = labels

In [32]:
# 針對不同類型的感染 分別植入DGP

group_setting = {

    "Virus": (1,20),

    "Gram+": (21,40),

    "Gram-": (41,60),

    "Non": (61,80)

}

for label, (start, end) in group_setting.items():

    mask = expression_df["Label"] == label

    for i in range(start, end, 2):

        geneA = f"Gene{i}"
        geneB = f"Gene{i+1}"

        noise = np.random.normal(loc=2.0, scale=0.3, size=mask.sum()) # 為了gene pair，第一個gene與第二個gene的大小比較由noise決定

        expression_df.loc[mask, geneA] = (
            expression_df.loc[mask, geneB] + noise
        )

In [ ]:
# expression_df.to_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\simulation_data_for_Biological_functions_of_the_DGPs\expression_df.csv", index=True)

In [35]:
pathway_db = {}

pathway_db["Virus_Core"] = [
    f"Gene{i}"
    for i in range(1,21) # 預設gene 1 ~ 20屬於virus感染路徑相關的gene
]

pathway_db["GramPos_Core"] = [
    f"Gene{i}"
    for i in range(21,41) # 預設gene 21 ~ 40屬於革蘭氏陽性菌感染路徑相關的gene
]

pathway_db["GramNeg_Core"] = [
    f"Gene{i}"
    for i in range(41,61) # 預設gene 41 ~ 60屬於革蘭氏陰性菌感染路徑相關的gene
]

pathway_db["Non_Core"] = [
    f"Gene{i}"
    for i in range(61,81) # 預設gene 61 ~ 80屬於未感染路徑相關的gene
]

In [42]:
n_random = 6 # 設定六種隨機的生理路徑

for p in range(n_random):

    size = np.random.randint(12,21) # 六種生理路徑，每一種隨機設定有幾個gene參與

    genes = np.random.choice(
        gene_names,
        size=size,
        replace=False 
    )

    pathway_db[f"Random_Pathway{p+1}"] = list(genes)

In [ ]:
used = set()

for genes in pathway_db.values():

    used.update(genes) 

unused = set(gene_names) - used

random_pathways = [
    p
    for p in pathway_db
    if "Random" in p
]

for gene in unused: # 未參與到任何一種生理機制的gene，隨機分配給Ramdom的生理機制

    p = np.random.choice(random_pathways)

    pathway_db[p].append(gene)

In [64]:
for gene in gene_names:

    extra = np.random.randint(0,3) # 決定這個gene要額外加入幾個pathway

    available = []

    for pathway in random_pathways: # 只選取ramdom pathway

        if gene not in pathway_db[pathway]:

            available.append(pathway)

    if len(available)==0: # 如果這個基因已經出現在所有ramdom pathway，就略過
        continue

    selected = np.random.choice(
        available,
        size=min(extra,len(available)),
        replace=False
    )

    for pathway in selected:

        pathway_db[pathway].append(gene)

In [65]:
print(type(pathway_db))
print(len(pathway_db))

<class 'dict'>
10


In [66]:
print(pathway_db.keys())

dict_keys(['Virus_Core', 'GramPos_Core', 'GramNeg_Core', 'Non_Core', 'Random_Pathway1', 'Random_Pathway2', 'Random_Pathway3', 'Random_Pathway4', 'Random_Pathway5', 'Random_Pathway6'])


In [67]:
print(pathway_db["Virus_Core"])

['Gene1', 'Gene2', 'Gene3', 'Gene4', 'Gene5', 'Gene6', 'Gene7', 'Gene8', 'Gene9', 'Gene10', 'Gene11', 'Gene12', 'Gene13', 'Gene14', 'Gene15', 'Gene16', 'Gene17', 'Gene18', 'Gene19', 'Gene20']


In [68]:
for k, v in pathway_db.items():
    print(k, len(v))

Virus_Core 20
GramPos_Core 20
GramNeg_Core 20
Non_Core 20
Random_Pathway1 34
Random_Pathway2 28
Random_Pathway3 39
Random_Pathway4 36
Random_Pathway5 32
Random_Pathway6 26


In [69]:
used_genes = set()

for genes in pathway_db.values():
    used_genes.update(genes)

print("used genes:", len(used_genes))

used genes: 100


In [70]:
import json

# with open(r"C:\Users\USER\Desktop\資訊所實習\計畫\simulation_data_for_Biological_functions_of_the_DGPs\pathway_db.json", "w") as f:
#    json.dump(pathway_db, f, indent=4)

In [71]:
from collections import Counter

gene_count = Counter()

for genes in pathway_db.values():
    gene_count.update(genes)

counts = list(gene_count.values())

print("mean:", np.mean(counts))
print("min:", np.min(counts))
print("max:", np.max(counts))

mean: 2.75
min: 1
max: 6


In [72]:
gene_to_pathway = {}

for pathway, genes in pathway_db.items():
    for g in genes:
        gene_to_pathway.setdefault(g, []).append(pathway)

# with open(r"C:\Users\USER\Desktop\資訊所實習\計畫\simulation_data_for_Biological_functions_of_the_DGPs\gene_to_pathway.json", "w") as f:
#    json.dump(gene_to_pathway, f, indent=4)